In [5]:
# SECTION 1 - INSTALL PACKAGES

# This notebook is the simple full-run version.
# It fetches API data, creates CSV outputs, runs fuzzy matching (Levenshtein Distance),
# runs TF-IDF cosine similarity, creates automated validation evidence,
# and uploads the final outputs to SQL Server.

!pip -q install thefuzz python-Levenshtein scikit-learn tqdm sqlalchemy pyodbc

# SQL Server driver for Colab.
# This is needed only if RUN_SQL_UPLOAD = True in Section 2.
!curl -sSL https://packages.microsoft.com/keys/microsoft.asc | gpg --dearmor | sudo tee /usr/share/keyrings/microsoft-prod.gpg > /dev/null
!echo "deb [arch=amd64,arm64,armhf signed-by=/usr/share/keyrings/microsoft-prod.gpg] https://packages.microsoft.com/ubuntu/22.04/prod jammy main" | sudo tee /etc/apt/sources.list.d/mssql-release.list > /dev/null
!sudo apt-get update -qq
!sudo ACCEPT_EULA=Y apt-get install -y -qq msodbcsql18 unixodbc-dev

print("Package and SQL driver installation completed.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Package and SQL driver installation completed.


In [6]:
# SECTION 2 - IMPORTS, DRIVE FOLDER, AND SETTINGS

from google.colab import drive
import os
import re
import json
import glob
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

from thefuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Google Drive project folder
DRIVE_ROOT = "/content/drive"
BASE_DIR = "/content/drive/MyDrive/LD6053-dissertation"

if not os.path.isdir(os.path.join(DRIVE_ROOT, "MyDrive")):
    try:
        drive.mount(DRIVE_ROOT, force_remount=True, timeout_ms=300000)
    except Exception as e:
        raise RuntimeError("Google Drive did not mount. Check Drive permission and run again.") from e
else:
    print("Google Drive is working.")

RUN_DATE = datetime.now().strftime("%Y%m%d")
RUN_DIR = os.path.join(BASE_DIR, RUN_DATE)
os.makedirs(RUN_DIR, exist_ok=True)

# ==============================================================
# SIMPLE FULL-RUN SETTINGS
# ==============================================================
# Change only this value for testing:
#   1 = quick test, using one page from each API where page limits apply.
#   100 = larger run.
#   0 = no artificial page limit.
PAGE_TEST_LIMIT = 1

# This notebook only runs the full pipeline.
RUN_MODE = "FULL_SINGLE_PAGE_TEST_V22"
FAST_DEMO_MODE = False
USE_EXISTING_CSV_IF_AVAILABLE = False
REFETCH_API_DATA = True
FORCE_REBUILD_MATCHING = True
RUN_SQL_UPLOAD = True
SHOW_PRESENTATION_SUMMARY = True

# Earlier script settings kept for the real run.
CONTRACTS_MONTHS_TO_FETCH = 12
CONTRACTS_PAGES_PER_MONTH_ORIGINAL = 50
GTR_PAGES_TO_FETCH_ORIGINAL = 100

# Matching thresholds
FUZZY_THRESHOLD = 90
ML_THRESHOLD_PERCENT = 75.0

# Runtime optimisation for fuzzy matching.
# This keeps fuzzy matching but compares against a TF-IDF shortlist instead of every sponsor name.
# It makes the notebook safer to run before the presentation while keeping the same pipeline stages.
USE_TFIDF_SHORTLIST_FOR_FUZZY = True
FUZZY_CANDIDATE_SHORTLIST = 25

# Automated validation thresholds
AUTO_ACCEPT_FUZZY_THRESHOLD = 90
AUTO_ACCEPT_ML_THRESHOLD_PERCENT = 75.0

# Source controls
RUN_CONTRACTS_FINDER = True
RUN_GTR = True

# SQL upload settings
SQL_SPONSOR_MODE = "MATCHED"  # MATCHED or FULL
SQL_INSERT_CHUNK_SIZE = 500    # larger chunks are faster for small/medium uploads
UPLOAD_BRONZE_SILVER_TO_SQL = False
USE_AUTOMATED_VALIDATION = True
RESET_VALIDATION_TABLE_ON_RUN = True
SQL_TEXT_MAX_LENGTH = 1000
SQL_DROP_RAW_JSON_FOR_SQL = True

# API settings
CONTRACTS_BASE_URL = "https://www.contractsfinder.service.gov.uk/Published/Notices/OCDS/Search"
GTR_BASE_URL = "https://gtr.ukri.org/gtr/api/projects"
GTR_HEADERS = {"Accept": "application/vnd.rcuk.gtr.json-v7"}
GTR_ORG_CACHE_FILE = os.path.join(BASE_DIR, "gtr_organisation_name_cache.csv")

# SQL Server settings
SQL_SERVER = r"sql.bsite.net\MSSQL2016"
SQL_DATABASE = "jorgetomaschabrillon_LD6053dissertation"
SQL_USERNAME = "jorgetomaschabrillon_LD6053dissertation"
SQL_PASSWORD = "Topito1990**"

# Sponsor Register file.
# The newest matching Home Office CSV in the project folder is selected automatically.
# This is useful when a newer Sponsor Register file is added before a presentation run.
def find_latest_sponsor_register_file():
    sponsor_patterns = [
        os.path.join(BASE_DIR, "*Worker_and_Temporary_Worker*.csv"),
        os.path.join(BASE_DIR, "SP_*Worker*.csv")
    ]

    candidates = []
    for pattern in sponsor_patterns:
        candidates.extend(glob.glob(pattern))

    candidates = sorted(set(candidates))
    if not candidates:
        return None

    candidates.sort(key=lambda file_path: os.path.getmtime(file_path), reverse=True)
    return candidates[0]


SPONSOR_FILE = find_latest_sponsor_register_file()


def page_limit(original_value):
    limit = int(PAGE_TEST_LIMIT)
    if limit == 0:
        return None
    return min(limit, int(original_value))


CONTRACTS_PAGES_PER_MONTH = page_limit(CONTRACTS_PAGES_PER_MONTH_ORIGINAL)
GTR_PAGES_TO_FETCH = page_limit(GTR_PAGES_TO_FETCH_ORIGINAL)

print("Project folder:", BASE_DIR)
print("Run folder:", RUN_DIR)
print("Run mode:", RUN_MODE)
print("Fast demo mode:", FAST_DEMO_MODE)
print("Use existing CSV outputs:", USE_EXISTING_CSV_IF_AVAILABLE)
print("Refetch API data:", REFETCH_API_DATA)
print("Force rebuild matching:", FORCE_REBUILD_MATCHING)
print("Run SQL upload:", RUN_SQL_UPLOAD)
print("Show presentation summary:", SHOW_PRESENTATION_SUMMARY)
print("Sponsor file detected:", SPONSOR_FILE)
if SPONSOR_FILE:
    print("Sponsor file modified:", datetime.fromtimestamp(os.path.getmtime(SPONSOR_FILE)).strftime("%Y-%m-%d %H:%M:%S"))
print("Contracts Finder months:", CONTRACTS_MONTHS_TO_FETCH)
print("Contracts Finder pages per month:", "NO LIMIT" if CONTRACTS_PAGES_PER_MONTH is None else CONTRACTS_PAGES_PER_MONTH)
print("GTR pages:", "NO LIMIT" if GTR_PAGES_TO_FETCH is None else GTR_PAGES_TO_FETCH)


Google Drive is working.
Project folder: /content/drive/MyDrive/LD6053-dissertation
Run folder: /content/drive/MyDrive/LD6053-dissertation/20260730
Run mode: FULL_SINGLE_PAGE_TEST_V22
Fast demo mode: False
Use existing CSV outputs: False
Refetch API data: True
Force rebuild matching: True
Run SQL upload: True
Show presentation summary: True
Sponsor file detected: /content/drive/MyDrive/LD6053-dissertation/!SP_-_Worker_and_Temporary_Worker_Web_Register_-_2026-07-23.csv
Sponsor file modified: 2026-07-23 12:03:54
Contracts Finder months: 12
Contracts Finder pages per month: 1
GTR pages: 1


In [7]:
# SECTION 3 - GENERAL HELPER FUNCTIONS

def current_timestamp():
    """Return a timestamp suitable for SQL Server datetime columns."""
    return datetime.now(timezone.utc).replace(tzinfo=None)


def runtime_row_count(obj):
    """Return row count for timing logs."""
    try:
        if obj is None:
            return 0
        if isinstance(obj, tuple):
            obj = obj[0]
        if hasattr(obj, "__len__"):
            return int(len(obj))
    except Exception:
        pass
    return None


RUNTIME_LOG = []


def log_runtime_step(step_name, start_time, status="SUCCESS", rows=None, notes=""):
    """Save timing information for dissertation evidence and optimisation."""
    end_time = time.perf_counter()
    duration_seconds = round(end_time - start_time, 2)
    RUNTIME_LOG.append({
        "RunDate": RUN_DATE,
        "RunMode": RUN_MODE,
        "PageTestLimit": PAGE_TEST_LIMIT,
        "StepName": step_name,
        "Status": status,
        "Rows": rows,
        "DurationSeconds": duration_seconds,
        "StartedAt": datetime.fromtimestamp(start_time).strftime("%Y-%m-%d %H:%M:%S"),
        "FinishedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Notes": str(notes)[:500]
    })
    print(f"[TIME] {step_name}: {duration_seconds} seconds | {status}")


def run_timed_step(step_name, function_to_run):
    """Run one pipeline step and log how long it takes."""
    step_start = time.perf_counter()
    try:
        result = function_to_run()
        log_runtime_step(step_name, step_start, "SUCCESS", runtime_row_count(result), "")
        return result
    except Exception as e:
        log_runtime_step(step_name, step_start, "FAILED", None, str(e))
        save_runtime_log()
        raise


def save_runtime_log():
    """Save runtime log to the dated Google Drive run folder."""
    if not RUNTIME_LOG:
        return None
    runtime_df = pd.DataFrame(RUNTIME_LOG)
    runtime_path = os.path.join(RUN_DIR, "LD6053_RUNTIME_LOG.csv")
    os.makedirs(os.path.dirname(runtime_path), exist_ok=True)
    runtime_df.to_csv(runtime_path, index=False, encoding="utf-8-sig")
    print("Runtime log saved:", runtime_path)
    return runtime_path


def make_output_path(source, layer, method=None):
    """Build the output filename inside the dated run folder."""
    source = str(source).upper()
    layer = str(layer).upper()

    if method:
        method = str(method).upper().replace("FUZZY", "FUZ")
        filename = f"{source}_{layer}_{method}.csv"
    else:
        filename = f"{source}_{layer}.csv"

    return os.path.join(RUN_DIR, filename)


def clean_organisation_name(value):
    """Standardise organisation names before comparison."""
    if pd.isna(value):
        return ""
    value = str(value).upper().strip()
    value = value.replace("&", " AND ")
    value = re.sub(r"[\.,;:'\"\(\)\[\]\{\}/\\\-]", " ", value)
    value = re.sub(r"\bLTD\b", "LIMITED", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def safe_get(dictionary, path, default=None):
    """Read a nested value safely from a dictionary/list structure."""
    current = dictionary
    for key in path:
        try:
            if isinstance(current, list):
                current = current[key]
            else:
                current = current.get(key, default)
        except Exception:
            return default
        if current is None:
            return default
    return current


def add_metadata_columns(df, layer, source):
    """Add audit columns used in the database and dashboard."""
    df = df.copy()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if "record_id" not in df.columns:
        df.insert(0, "record_id", range(1, len(df) + 1))

    df["source_system"] = source
    df["medallion_layer"] = layer
    df["inserted_date"] = now
    df["modified_date"] = now

    if "matching_method" not in df.columns:
        df["matching_method"] = None
    if "matching_accuracy_percent" not in df.columns:
        df["matching_accuracy_percent"] = None

    return df


def save_csv(df, source, layer, method=None):
    """Save a dataframe in the dated Google Drive run folder."""
    output_path = make_output_path(source, layer, method)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df):,} rows -> {output_path}")
    return output_path


def find_existing_csv(source, layer, method=None):
    """Find an existing output CSV in the current run folder, or the newest older run folder."""
    expected_name = os.path.basename(make_output_path(source, layer, method))
    direct_path = os.path.join(RUN_DIR, expected_name)
    if os.path.exists(direct_path):
        return direct_path

    candidates = []
    for folder in glob.glob(os.path.join(BASE_DIR, "*")):
        if os.path.isdir(folder):
            possible = os.path.join(folder, expected_name)
            if os.path.exists(possible):
                candidates.append(possible)

    if not candidates:
        return None

    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]


def load_existing_csv(source, layer, method=None):
    """Load an existing output CSV if it is available."""
    path = find_existing_csv(source, layer, method)
    if path is None:
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded existing {source}_{layer}{'_' + method if method else ''}: {len(df):,} rows <- {path}")
    return df, path


def get_or_create_csv(source, layer, create_function, method=None, use_existing=True, force_create=False):
    """Use an existing CSV when possible; otherwise create and save a new one."""
    if use_existing and not force_create:
        df, path = load_existing_csv(source, layer, method)
        if df is not None:
            return df, path

    df = create_function()
    path = save_csv(df, source, layer, method)
    return df, path


def normalise_sql_column_names(df):
    """Make column names safe for SQL Server."""
    df = df.copy()
    new_columns = []
    used = set()

    for col in df.columns:
        clean = re.sub(r"[^A-Za-z0-9_]", "_", str(col).strip())
        clean = re.sub(r"_+", "_", clean).strip("_")
        if clean == "":
            clean = "column"
        if re.match(r"^\d", clean):
            clean = "c_" + clean

        base = clean
        number = 2
        while clean.lower() in used:
            clean = f"{base}_{number}"
            number += 1
        used.add(clean.lower())
        new_columns.append(clean)

    df.columns = new_columns
    return df


def prepare_for_sql(df):
    """Convert a dataframe into a safer SQL upload format."""
    df = normalise_sql_column_names(df).copy()

    if SQL_DROP_RAW_JSON_FOR_SQL:
        raw_cols = [c for c in df.columns if c.lower() in ["raw_release_json", "raw_project_json"]]
        if raw_cols:
            df = df.drop(columns=raw_cols)

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].astype(str).replace({"NaT": None})
        elif df[col].dtype == "object":
            df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else x)
            df[col] = df[col].apply(lambda x: x[:SQL_TEXT_MAX_LENGTH] if isinstance(x, str) and len(x) > SQL_TEXT_MAX_LENGTH else x)

    return df.where(pd.notnull(df), None)


def empty_dataframe(columns):
    return pd.DataFrame(columns=list(dict.fromkeys(columns)))


In [8]:
# SECTION 4 - HOME OFFICE SPONSOR REGISTER

def load_sponsor_lookup():
    """Load the Home Office Sponsor Register used as the reference dataset."""
    if SPONSOR_FILE is None:
        raise FileNotFoundError(
            "Sponsor Register CSV not found. Put the Worker_and_Temporary_Worker CSV inside: " + BASE_DIR
        )

    df = pd.read_csv(SPONSOR_FILE, low_memory=False)
    if "Organisation Name" not in df.columns:
        raise ValueError("Sponsor Register must contain a column called Organisation Name.")

    df = df.copy()
    df["sponsor_clean_name"] = df["Organisation Name"].apply(clean_organisation_name)
    df = df[df["sponsor_clean_name"] != ""].copy()

    agg_dict = {}
    for col in df.columns:
        if col == "Route":
            agg_dict[col] = lambda x: " / ".join(sorted(set(str(v) for v in x.dropna().unique())))
        elif col != "sponsor_clean_name":
            agg_dict[col] = "first"

    lookup = df.groupby("sponsor_clean_name", as_index=False).agg(agg_dict)
    lookup.insert(0, "sponsor_id", range(1, len(lookup) + 1))

    rename_map = {col: f"sponsor_{col}" for col in lookup.columns if col not in ["sponsor_id", "sponsor_clean_name"]}
    lookup = lookup.rename(columns=rename_map)
    lookup = add_metadata_columns(lookup, "REFERENCE", "HOME")

    sponsor_names = lookup["sponsor_clean_name"].tolist()
    print(f"Loaded {len(sponsor_names):,} unique sponsor organisations.")
    return lookup, sponsor_names


def attach_sponsor_columns(result_df, sponsor_lookup):
    """Attach sponsor details to matched records."""
    if result_df.empty:
        base_cols = list(result_df.columns)
        sponsor_cols = [c for c in sponsor_lookup.columns if c not in base_cols]
        return empty_dataframe(base_cols + sponsor_cols)

    return result_df.merge(
        sponsor_lookup,
        left_on="matched_sponsor_clean",
        right_on="sponsor_clean_name",
        how="left"
    )


In [9]:
# SECTION 5 - MATCHING FUNCTIONS

def gold_empty_frame(df_silver, sponsor_lookup):
    base_cols = list(df_silver.columns)
    match_cols = [
        "source_organisation_original",
        "source_organisation_clean",
        "matched_sponsor_clean",
        "matching_step",
        "matching_method",
        "matching_accuracy_percent"
    ]
    sponsor_cols = [c for c in sponsor_lookup.columns if c not in base_cols + match_cols]
    return empty_dataframe(base_cols + match_cols + sponsor_cols)


def build_tfidf_shortlist(unique_names, sponsor_names, neighbours=25):
    """Return likely sponsor-name candidates for each source name."""
    if not unique_names or not sponsor_names:
        return {}

    neighbour_count = min(int(neighbours), len(sponsor_names))
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    sponsor_matrix = vectorizer.fit_transform(sponsor_names)
    source_matrix = vectorizer.transform(unique_names)

    nn = NearestNeighbors(n_neighbors=neighbour_count, metric="cosine")
    nn.fit(sponsor_matrix)
    distances, indices = nn.kneighbors(source_matrix)

    shortlist = {}
    for row_index, source_clean in enumerate(unique_names):
        shortlist[source_clean] = [sponsor_names[int(i)] for i in indices[row_index]]
    return shortlist


def create_fuzzy_gold(df_silver, source_org_col, sponsor_lookup, sponsor_names, source_label):
    """Create a Gold output using token-based fuzzy matching."""
    if df_silver.empty or source_org_col not in df_silver.columns:
        return gold_empty_frame(df_silver, sponsor_lookup)

    df = df_silver.copy()
    df["source_organisation_original"] = df[source_org_col]
    df["source_organisation_clean"] = df[source_org_col].apply(clean_organisation_name)
    df = df[df["source_organisation_clean"] != ""].copy()

    if df.empty:
        return gold_empty_frame(df_silver, sponsor_lookup)

    unique_names = df["source_organisation_clean"].drop_duplicates().tolist()
    sponsor_name_set = set(sponsor_names)
    match_rows = []

    non_exact_names = [name for name in unique_names if name not in sponsor_name_set]
    shortlist = {}
    if USE_TFIDF_SHORTLIST_FOR_FUZZY and non_exact_names:
        shortlist = build_tfidf_shortlist(
            non_exact_names,
            sponsor_names,
            neighbours=FUZZY_CANDIDATE_SHORTLIST
        )

    for source_clean in tqdm(unique_names, desc=f"{source_label} fuzzy matching"):
        if source_clean in sponsor_name_set:
            matched_clean = source_clean
            score = 100.0
            step = "EXACT"
            method = "EXACT"
        else:
            candidates = shortlist.get(source_clean, sponsor_names)
            match = process.extractOne(source_clean, candidates, scorer=fuzz.token_sort_ratio)
            if match and match[1] >= FUZZY_THRESHOLD:
                matched_clean = match[0]
                score = float(match[1])
                step = "TOKEN_SORT_RATIO"
                method = "FUZZY"
            else:
                continue

        match_rows.append({
            "source_organisation_clean": source_clean,
            "matched_sponsor_clean": matched_clean,
            "matching_step": step,
            "matching_method": method,
            "matching_accuracy_percent": round(score, 2)
        })

    if not match_rows:
        return gold_empty_frame(df_silver, sponsor_lookup)

    match_df = pd.DataFrame(match_rows)
    result = df.merge(match_df, on="source_organisation_clean", how="inner")
    result = attach_sponsor_columns(result, sponsor_lookup)
    result = add_metadata_columns(result, "GOLD", source_label)
    return result


def create_ml_gold(df_silver, source_org_col, sponsor_lookup, sponsor_names, source_label):
    """Create a Gold output using TF-IDF character n-grams and nearest-neighbour similarity. Unique names are compared once."""
    if df_silver.empty or source_org_col not in df_silver.columns or not sponsor_names:
        return gold_empty_frame(df_silver, sponsor_lookup)

    df = df_silver.copy()
    df["source_organisation_original"] = df[source_org_col]
    df["source_organisation_clean"] = df[source_org_col].apply(clean_organisation_name)
    df = df[df["source_organisation_clean"] != ""].copy()

    if df.empty:
        return gold_empty_frame(df_silver, sponsor_lookup)

    unique_names = df["source_organisation_clean"].drop_duplicates().tolist()

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    sponsor_matrix = vectorizer.fit_transform(sponsor_names)
    source_matrix = vectorizer.transform(unique_names)

    nn = NearestNeighbors(n_neighbors=1, metric="cosine")
    nn.fit(sponsor_matrix)
    distances, indices = nn.kneighbors(source_matrix)

    match_rows = []
    sponsor_name_set = set(sponsor_names)

    for row_index, source_clean in enumerate(tqdm(unique_names, desc=f"{source_label} TF-IDF matching")):
        if source_clean in sponsor_name_set:
            matched_clean = source_clean
            score_percent = 100.0
            step = "EXACT"
            method = "EXACT"
        else:
            best_index = int(indices[row_index][0])
            similarity = 1 - float(distances[row_index][0])
            score_percent = similarity * 100
            matched_clean = sponsor_names[best_index]
            step = "TFIDF_COSINE"
            method = "ML"

            if score_percent < ML_THRESHOLD_PERCENT:
                continue

        match_rows.append({
            "source_organisation_clean": source_clean,
            "matched_sponsor_clean": matched_clean,
            "matching_step": step,
            "matching_method": method,
            "matching_accuracy_percent": round(score_percent, 2)
        })

    if not match_rows:
        return gold_empty_frame(df_silver, sponsor_lookup)

    match_df = pd.DataFrame(match_rows)
    result = df.merge(match_df, on="source_organisation_clean", how="inner")
    result = attach_sponsor_columns(result, sponsor_lookup)
    result = add_metadata_columns(result, "GOLD", source_label)
    return result


def has_text(value):
    return pd.notna(value) and str(value).strip() != ""


def score_value(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except Exception:
        return None


def automatic_match_decision(source_org, fuzzy_match, ml_match, fuzzy_score, ml_score):
    """Return an automated confidence decision for one matched organisation."""
    source_text = str(source_org).strip() if has_text(source_org) else ""
    fuzzy_text = str(fuzzy_match).strip() if has_text(fuzzy_match) else ""
    ml_text = str(ml_match).strip() if has_text(ml_match) else ""

    fuzzy_has_match = fuzzy_text != ""
    ml_has_match = ml_text != ""
    fuzzy_score_value = score_value(fuzzy_score)
    ml_score_value = score_value(ml_score)

    if not fuzzy_has_match and not ml_has_match:
        return None, None, None, "UNMATCHED", "No sponsor match was found by either method.", 0

    exact_match = (
        source_text != ""
        and ((fuzzy_has_match and fuzzy_text == source_text) or (ml_has_match and ml_text == source_text))
    )

    if exact_match:
        match_text = fuzzy_text if fuzzy_has_match else ml_text
        return "Exact", match_text, 100.0, "ACCEPTED", "Exact cleaned-name match to the Sponsor Register.", 1

    methods_agree = fuzzy_has_match and ml_has_match and fuzzy_text == ml_text

    if methods_agree:
        fuzzy_ok = fuzzy_score_value is not None and fuzzy_score_value >= AUTO_ACCEPT_FUZZY_THRESHOLD
        ml_ok = ml_score_value is not None and ml_score_value >= AUTO_ACCEPT_ML_THRESHOLD_PERCENT
        best_score = max(
            fuzzy_score_value if fuzzy_score_value is not None else -1,
            ml_score_value if ml_score_value is not None else -1
        )

        if fuzzy_ok and ml_ok:
            return "Fuzzy+ML", fuzzy_text, best_score, "ACCEPTED", "Fuzzy and TF-IDF selected the same sponsor above the acceptance thresholds.", 1

        return "Fuzzy+ML", fuzzy_text, best_score, "EXCLUDED_LOW_CONFIDENCE", "Both methods selected the same sponsor, but one or both scores were below the acceptance thresholds.", 0

    if fuzzy_has_match and ml_has_match:
        if (fuzzy_score_value or -1) >= (ml_score_value or -1):
            return "Fuzzy", fuzzy_text, fuzzy_score_value, "EXCLUDED_METHOD_DISAGREEMENT", "Fuzzy and TF-IDF selected different sponsors.", 0
        return "ML", ml_text, ml_score_value, "EXCLUDED_METHOD_DISAGREEMENT", "Fuzzy and TF-IDF selected different sponsors.", 0

    if fuzzy_has_match:
        return "Fuzzy", fuzzy_text, fuzzy_score_value, "EXCLUDED_SINGLE_METHOD", "Only fuzzy matching found a sponsor, so the match was excluded from automated dashboard outputs.", 0

    return "ML", ml_text, ml_score_value, "EXCLUDED_SINGLE_METHOD", "Only TF-IDF cosine similarity found a sponsor, so the match was excluded from automated dashboard outputs.", 0


def best_unique_matches(df, match_col_name, score_col_name):
    """Keep one match row per cleaned source organisation for comparison evidence."""
    if df.empty or "source_organisation_clean" not in df.columns:
        return empty_dataframe(["source_organisation_clean", match_col_name, score_col_name])

    temp = df[["source_organisation_clean", "matched_sponsor_clean", "matching_accuracy_percent"]].copy()
    temp = temp.rename(columns={
        "matched_sponsor_clean": match_col_name,
        "matching_accuracy_percent": score_col_name
    })
    temp[score_col_name] = pd.to_numeric(temp[score_col_name], errors="coerce")
    temp = temp.sort_values(by=["source_organisation_clean", score_col_name], ascending=[True, False])
    temp = temp.drop_duplicates(subset=["source_organisation_clean"], keep="first")
    return temp


def create_matching_comparison(fuzzy_df, ml_df, source_label):
    """Compare fuzzy and TF-IDF matching outputs and assign automated validation status."""
    comparison_cols = [
        "source_system",
        "source_organisation_clean",
        "fuzzy_match",
        "fuzzy_score_percent",
        "ml_match",
        "ml_score_percent",
        "methods_agree",
        "fuzzy_found_match",
        "ml_found_match",
        "recommended_method",
        "recommended_match",
        "recommended_score_percent",
        "auto_validation_status",
        "auto_validation_reason",
        "dashboard_eligible"
    ]

    fuzzy_part = best_unique_matches(fuzzy_df, "fuzzy_match", "fuzzy_score_percent")
    ml_part = best_unique_matches(ml_df, "ml_match", "ml_score_percent")

    comparison = fuzzy_part.merge(ml_part, on="source_organisation_clean", how="outer")

    if comparison.empty:
        comparison = empty_dataframe(comparison_cols)
    else:
        comparison.insert(0, "source_system", source_label)
        comparison["methods_agree"] = (
            comparison["fuzzy_match"].fillna("").astype(str) == comparison["ml_match"].fillna("").astype(str)
        ) & comparison["fuzzy_match"].notna() & comparison["ml_match"].notna()
        comparison["fuzzy_found_match"] = comparison["fuzzy_match"].notna()
        comparison["ml_found_match"] = comparison["ml_match"].notna()

        decisions = comparison.apply(
            lambda row: automatic_match_decision(
                row.get("source_organisation_clean"),
                row.get("fuzzy_match"),
                row.get("ml_match"),
                row.get("fuzzy_score_percent"),
                row.get("ml_score_percent")
            ),
            axis=1,
            result_type="expand"
        )
        decisions.columns = [
            "recommended_method",
            "recommended_match",
            "recommended_score_percent",
            "auto_validation_status",
            "auto_validation_reason",
            "dashboard_eligible"
        ]
        comparison = pd.concat([comparison, decisions], axis=1)
        comparison["methods_agree"] = comparison["methods_agree"].astype(int)
        comparison["fuzzy_found_match"] = comparison["fuzzy_found_match"].astype(int)
        comparison["ml_found_match"] = comparison["ml_found_match"].astype(int)
        comparison["dashboard_eligible"] = comparison["dashboard_eligible"].astype(int)
        comparison = comparison[comparison_cols]

    comparison = add_metadata_columns(comparison, "EVALUATION", source_label)
    return comparison


In [10]:
# SECTION 6 - CONTRACTS FINDER PIPELINE

def fetch_contracts_finder_bronze():
    """Extract Contracts Finder award data into the Bronze layer."""
    all_rows = []

    for month_index in range(CONTRACTS_MONTHS_TO_FETCH):
        start_date = (datetime.now() - timedelta(days=(month_index + 1) * 30)).strftime('%Y-%m-%d')
        end_date = (datetime.now() - timedelta(days=month_index * 30)).strftime('%Y-%m-%d')

        if CONTRACTS_PAGES_PER_MONTH is None:
            page_iter = range(1, 10**9)
        else:
            page_iter = range(1, CONTRACTS_PAGES_PER_MONTH + 1)

        for page in tqdm(page_iter, desc=f"Contracts Finder month {month_index + 1}"):
            params = {
                "type": "award",
                "stages": "award",
                "publishedFrom": start_date,
                "publishedTo": end_date,
                "page": page,
                "limit": 100
            }

            try:
                response = requests.get(CONTRACTS_BASE_URL, params=params, timeout=25)
                if response.status_code != 200:
                    break

                payload = response.json()
                releases = payload.get("releases", [])
                if not releases:
                    results = payload.get("results", [])
                    releases = [item.get("releases", [None])[0] for item in results if item.get("releases")]

                if not releases:
                    break

                for rel in releases:
                    tender = rel.get("tender", {}) or {}
                    awards = rel.get("awards", []) or []

                    for award in awards:
                        suppliers = award.get("suppliers", []) or []
                        for supplier in suppliers:
                            supplier_name = supplier.get("name")
                            if not supplier_name:
                                continue

                            award_date = award.get("date")
                            if award_date and "T" in str(award_date):
                                award_date = str(award_date).split("T")[0]

                            row = {
                                "Contract_Title": tender.get("title"),
                                "Contract_Value": safe_get(award, ["value", "amount"], 0),
                                "Contract_Currency": safe_get(award, ["value", "currency"], None),
                                "Supplier_Name": supplier_name,
                                "Supplier_ID": supplier.get("id"),
                                "Award_Date": award_date,
                                "Award_ID": award.get("id"),
                                "OCID": rel.get("ocid"),
                                "Release_ID": rel.get("id"),
                                "Published_Date": rel.get("date"),
                                "API_Month_Start": start_date,
                                "API_Month_End": end_date,
                                "API_Page": page,
                                "raw_release_json": json.dumps(rel, ensure_ascii=False)
                            }
                            all_rows.append(row)

                time.sleep(0.3)

            except Exception as e:
                print("Contracts Finder extraction error:", e)
                break

    df = pd.DataFrame(all_rows).drop_duplicates()
    df = add_metadata_columns(df, "BRONZE", "CF")
    return df


def clean_contracts_finder_silver(df_bronze):
    """Clean Contracts Finder Bronze data into the Silver layer."""
    if df_bronze.empty:
        columns = [
            "Contract_Title", "Contract_Value", "Contract_Currency", "Supplier_Name", "Supplier_ID",
            "Award_Date", "Award_ID", "OCID", "Release_ID", "Published_Date",
            "API_Month_Start", "API_Month_End", "API_Page", "raw_release_json",
            "Supplier_Name_Clean"
        ]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "CF")

    df = df_bronze.copy()
    df = df.dropna(subset=["Supplier_Name"])
    df["Supplier_Name"] = df["Supplier_Name"].astype(str).str.upper().str.strip()
    df["Supplier_Name_Clean"] = df["Supplier_Name"].apply(clean_organisation_name)
    df["Contract_Value"] = pd.to_numeric(df["Contract_Value"], errors="coerce").fillna(0)
    df["Award_Date"] = pd.to_datetime(df["Award_Date"], errors="coerce").dt.date
    df = df.drop_duplicates()
    df = df.sort_values(by="Award_Date", ascending=False, na_position="last")
    df = add_metadata_columns(df, "SILVER", "CF")
    return df


In [11]:
# SECTION 7 - GATEWAY TO RESEARCH PIPELINE

def fetch_gtr_bronze():
    """Extract Gateway to Research project data into the Bronze layer."""
    all_rows = []

    try:
        init_res = requests.get(GTR_BASE_URL, headers=GTR_HEADERS, params={"s": 100, "p": 1}, timeout=20)
        total_pages = int(init_res.json().get("totalPages", 1500))
    except Exception:
        total_pages = 1500

    # Page 1 is used first so the quick test is more likely to collect usable active records.
    if GTR_PAGES_TO_FETCH is None:
        target_pages = range(1, total_pages + 1)
    else:
        target_pages = range(1, min(total_pages, GTR_PAGES_TO_FETCH) + 1)

    for page in tqdm(target_pages, desc="GTR project pages"):
        try:
            res = requests.get(GTR_BASE_URL, headers=GTR_HEADERS, params={"s": 100, "p": page}, timeout=20)
            projects = res.json().get("project", [])
            if not projects:
                continue

            for p in projects:
                links = p.get("links", {}).get("link", [])
                if isinstance(links, dict):
                    links = [links]

                org_url = "Unknown"
                for link in links:
                    if isinstance(link, dict) and link.get("rel") == "LEAD_ORG":
                        org_url = link.get("href", "Unknown")
                        break

                fund_info = p.get("fund", {}) if isinstance(p.get("fund", {}), dict) else {}

                start_date = fund_info.get("start")
                if start_date and "T" in str(start_date):
                    start_date = str(start_date).split("T")[0]

                end_date = fund_info.get("end")
                if end_date and "T" in str(end_date):
                    end_date = str(end_date).split("T")[0]

                row = {
                    "project_id": p.get("id"),
                    "project_url": p.get("href"),
                    "title": p.get("title"),
                    "status": p.get("status"),
                    "grant_reference": p.get("grantReference"),
                    "grant_category": p.get("grantCategory"),
                    "abstract_text": p.get("abstractText"),
                    "start_date": start_date,
                    "end_date": end_date,
                    "fund_type": fund_info.get("type"),
                    "value_pounds": fund_info.get("valuePounds"),
                    "funder_name": safe_get(fund_info, ["funder", "name"], None),
                    "funder_id": safe_get(fund_info, ["funder", "id"], None),
                    "org_url": org_url,
                    "API_Page": page,
                    "raw_project_json": json.dumps(p, ensure_ascii=False)
                }
                all_rows.append(row)

            time.sleep(0.2)

        except Exception as e:
            print("GTR extraction error:", e)
            break

    df = pd.DataFrame(all_rows).drop_duplicates()
    df = add_metadata_columns(df, "BRONZE", "GTR")
    return df


def load_gtr_org_cache():
    if not os.path.exists(GTR_ORG_CACHE_FILE):
        return {}
    try:
        cache_df = pd.read_csv(GTR_ORG_CACHE_FILE)
        return dict(zip(cache_df["org_url"].astype(str), cache_df["lead_organisation"].astype(str)))
    except Exception:
        return {}


def save_gtr_org_cache(cache):
    try:
        cache_df = pd.DataFrame({"org_url": list(cache.keys()), "lead_organisation": list(cache.values())})
        cache_df.to_csv(GTR_ORG_CACHE_FILE, index=False, encoding="utf-8-sig")
    except Exception as e:
        print("Could not save GTR organisation cache:", e)


def fetch_gtr_organisation_name(org_url):
    """Fetch the lead organisation name from a GTR organisation URL."""
    if not org_url or org_url == "Unknown":
        return "Unknown"
    try:
        secure_url = str(org_url).replace("http://", "https://")
        res = requests.get(secure_url, headers=GTR_HEADERS, timeout=10)
        if res.status_code == 200:
            return str(res.json().get("name", "Unknown")).upper().strip()
    except Exception:
        pass
    return "Unknown"


def clean_gtr_silver(df_bronze):
    """Clean GTR Bronze data into the Silver layer and add lead organisation names."""
    if df_bronze.empty:
        columns = [
            "project_id", "project_url", "title", "status", "grant_reference", "grant_category",
            "abstract_text", "start_date", "end_date", "fund_type", "value_pounds",
            "funder_name", "funder_id", "org_url", "API_Page", "raw_project_json",
            "lead_organisation", "lead_organisation_clean"
        ]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "GTR")

    df = df_bronze.copy()
    df = df[df["status"].astype(str).str.upper() == "ACTIVE"].copy()

    if df.empty:
        columns = list(df_bronze.columns) + ["lead_organisation", "lead_organisation_clean"]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "GTR")

    unique_urls = [u for u in df["org_url"].dropna().unique() if u != "Unknown"]
    url_to_name = load_gtr_org_cache()
    url_to_name["Unknown"] = "Unknown"

    missing_urls = [u for u in unique_urls if str(u) not in url_to_name]
    for url in tqdm(missing_urls, desc="Fetching GTR lead organisation names"):
        url_to_name[str(url)] = fetch_gtr_organisation_name(url)
        time.sleep(0.05)

    if missing_urls:
        save_gtr_org_cache(url_to_name)

    df["lead_organisation"] = df["org_url"].astype(str).map(url_to_name).fillna("Unknown")
    df = df[df["lead_organisation"].astype(str).str.upper() != "UNKNOWN"].copy()

    if df.empty:
        columns = list(df_bronze.columns) + ["lead_organisation", "lead_organisation_clean"]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "GTR")

    df["lead_organisation_clean"] = df["lead_organisation"].apply(clean_organisation_name)
    df["value_pounds"] = pd.to_numeric(df["value_pounds"], errors="coerce")
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce").dt.date
    df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce").dt.date
    df = df.drop_duplicates()
    df = df.sort_values(by="start_date", ascending=False, na_position="last")
    df = add_metadata_columns(df, "SILVER", "GTR")
    return df


In [12]:
# SECTION 8 - RUN FULL PIPELINE AND SAVE CSV OUTPUTS

created_files = []
tables_to_upload = {}

# Sponsor lookup. The Sponsor Register is the reference list used for matching.
sponsor_lookup, sponsor_names = run_timed_step(
    "Load Home Office Sponsor Register",
    load_sponsor_lookup
)
created_files.append(save_csv(sponsor_lookup, "HOME", "SPONSOR"))

if RUN_CONTRACTS_FINDER:
    print("\n================ CONTRACTS FINDER PIPELINE ================")

    cf_bronze = run_timed_step(
        "Contracts Finder API extraction - Bronze",
        fetch_contracts_finder_bronze
    )
    created_files.append(save_csv(cf_bronze, "CF", "BRONZE"))
    tables_to_upload["CFBronze"] = cf_bronze

    cf_silver = run_timed_step(
        "Contracts Finder cleaning - Silver",
        lambda: clean_contracts_finder_silver(cf_bronze)
    )
    created_files.append(save_csv(cf_silver, "CF", "SILVER"))
    tables_to_upload["CFSilver"] = cf_silver

    cf_gold_fuzzy = run_timed_step(
        "Contracts Finder fuzzy matching - Gold",
        lambda: create_fuzzy_gold(cf_silver, "Supplier_Name", sponsor_lookup, sponsor_names, "CF")
    )
    created_files.append(save_csv(cf_gold_fuzzy, "CF", "GOLD", "FUZ"))
    tables_to_upload["CFGoldFuz"] = cf_gold_fuzzy

    cf_gold_ml = run_timed_step(
        "Contracts Finder TF-IDF cosine matching - Gold",
        lambda: create_ml_gold(cf_silver, "Supplier_Name", sponsor_lookup, sponsor_names, "CF")
    )
    created_files.append(save_csv(cf_gold_ml, "CF", "GOLD", "ML"))
    tables_to_upload["CFGoldML"] = cf_gold_ml

    cf_comparison = run_timed_step(
        "Contracts Finder automated validation comparison",
        lambda: create_matching_comparison(cf_gold_fuzzy, cf_gold_ml, "CF")
    )
    created_files.append(save_csv(cf_comparison, "CF", "COMPARISON"))
    tables_to_upload["CFComparison"] = cf_comparison

if RUN_GTR:
    print("\n================ GATEWAY TO RESEARCH PIPELINE ================")

    gtr_bronze = run_timed_step(
        "Gateway to Research API extraction - Bronze",
        fetch_gtr_bronze
    )
    created_files.append(save_csv(gtr_bronze, "GTR", "BRONZE"))
    tables_to_upload["GTRBronze"] = gtr_bronze

    gtr_silver = run_timed_step(
        "Gateway to Research cleaning - Silver",
        lambda: clean_gtr_silver(gtr_bronze)
    )
    created_files.append(save_csv(gtr_silver, "GTR", "SILVER"))
    tables_to_upload["GTRSilver"] = gtr_silver

    gtr_gold_fuzzy = run_timed_step(
        "Gateway to Research fuzzy matching - Gold",
        lambda: create_fuzzy_gold(gtr_silver, "lead_organisation", sponsor_lookup, sponsor_names, "GTR")
    )
    created_files.append(save_csv(gtr_gold_fuzzy, "GTR", "GOLD", "FUZ"))
    tables_to_upload["GTRGoldFuz"] = gtr_gold_fuzzy

    gtr_gold_ml = run_timed_step(
        "Gateway to Research TF-IDF cosine matching - Gold",
        lambda: create_ml_gold(gtr_silver, "lead_organisation", sponsor_lookup, sponsor_names, "GTR")
    )
    created_files.append(save_csv(gtr_gold_ml, "GTR", "GOLD", "ML"))
    tables_to_upload["GTRGoldML"] = gtr_gold_ml

    gtr_comparison = run_timed_step(
        "Gateway to Research automated validation comparison",
        lambda: create_matching_comparison(gtr_gold_fuzzy, gtr_gold_ml, "GTR")
    )
    created_files.append(save_csv(gtr_comparison, "GTR", "COMPARISON"))
    tables_to_upload["GTRComparison"] = gtr_comparison

# Save timing information for the extraction, cleaning and matching stages.
runtime_path = save_runtime_log()
if runtime_path:
    created_files.append(runtime_path)

# ==============================================================
# PIPELINE EVIDENCE SUMMARY
# ==============================================================

def show_dataframe_preview(title, df, max_rows=5):
    print("\n" + title)
    if df is None:
        print("No dataframe available.")
        return
    print("Rows:", len(df), "| Columns:", len(df.columns))
    try:
        from IPython.display import display
        display(df.head(max_rows))
    except Exception:
        print(df.head(max_rows).to_string(index=False))


def show_comparison_summary(title, df):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    if df is None or df.empty:
        print("No comparison rows available.")
        return
    print("Total comparison rows:", len(df))
    if "auto_validation_status" in df.columns:
        print("\nAutomated validation status counts:")
        print(df["auto_validation_status"].fillna("MISSING").value_counts().to_string())
    if "dashboard_eligible" in df.columns:
        eligible = pd.to_numeric(df["dashboard_eligible"], errors="coerce").fillna(0).astype(int)
        print("\nDashboard eligible rows:", int(eligible.sum()))
        print("Excluded / not eligible rows:", int((eligible == 0).sum()))
    if "methods_agree" in df.columns:
        agree = pd.to_numeric(df["methods_agree"], errors="coerce").fillna(0).astype(int)
        if len(agree) > 0:
            print("Fuzzy and TF-IDF agreement rate:", round(float(agree.mean() * 100), 2), "%")

if SHOW_PRESENTATION_SUMMARY:
    print("\n" + "#" * 70)
    print("PIPELINE EVIDENCE SUMMARY")
    print("#" * 70)
    print("Run mode:", RUN_MODE)
    print("\nFiles created in this run:")
    for file_path in created_files:
        if file_path:
            print("-", file_path)

    if RUN_CONTRACTS_FINDER:
        show_dataframe_preview("Contracts Finder Bronze preview", globals().get("cf_bronze"), max_rows=3)
        show_dataframe_preview("Contracts Finder Silver preview", globals().get("cf_silver"), max_rows=3)
        show_comparison_summary("Contracts Finder automated validation evidence", globals().get("cf_comparison"))

    if RUN_GTR:
        show_dataframe_preview("Gateway to Research Bronze preview", globals().get("gtr_bronze"), max_rows=3)
        show_dataframe_preview("Gateway to Research Silver preview", globals().get("gtr_silver"), max_rows=3)
        show_comparison_summary("Gateway to Research automated validation evidence", globals().get("gtr_comparison"))

    print("\nPipeline summary completed.")


Loaded 126,582 unique sponsor organisations.
[TIME] Load Home Office Sponsor Register: 36.83 seconds | SUCCESS
Saved 126,582 rows -> /content/drive/MyDrive/LD6053-dissertation/20260730/HOME_SPONSOR.csv

================ CONTRACTS FINDER PIPELINE ================


Contracts Finder month 12: 100%|██████████| 1/1 [00:01<00:00,  1.64s/it]


[TIME] Contracts Finder API extraction - Bronze: 19.62 seconds | SUCCESS
Saved 1,497 rows -> /content/drive/MyDrive/LD6053-dissertation/20260730/CF_BRONZE.csv
[TIME] Contracts Finder cleaning - Silver: 0.05 seconds | SUCCESS
Saved 1,497 rows -> /content/drive/MyDrive/LD6053-dissertation/20260730/CF_SILVER.csv


CF fuzzy matching: 100%|██████████| 1180/1180 [00:00<00:00, 7892.20it/s]


[TIME] Contracts Finder fuzzy matching - Gold: 15.77 seconds | SUCCESS
Saved 581 rows -> /content/drive/MyDrive/LD6053-dissertation/20260730/CF_GOLD_FUZ.csv


...


Contracts Finder automated validation evidence
Total comparison rows: 612

Automated validation status counts:
auto_validation_status
ACCEPTED                        287
EXCLUDED_SINGLE_METHOD          257
EXCLUDED_METHOD_DISAGREEMENT     39
EXCLUDED_LOW_CONFIDENCE          29

Dashboard eligible rows: 287
Excluded / not eligible rows: 325
Fuzzy and TF-IDF agreement rate: 51.63 %

Gateway to Research Bronze preview
Rows: 100 | Columns: 23


,record_id,project_id,project_url,title,status,grant_reference,grant_category,abstract_text,start_date,end_date,...,funder_id,org_url,API_Page,raw_project_json,source_system,medallion_layer,inserted_date,modified_date,matching_method,matching_accuracy_percent
0,1,08DC2362-B8F7-4E0B-B1FC-001FB7AF5B94,http://gtr.ukri.org/gtr/api/projects/08DC2362-...,ENGLISH HERITAGE TRUST CONSERVATION AND HERITA...,Closed,None,Research Grant,The English Heritage Trust (EHT) cares for the...,None,None,...,None,http://gtr.ukri.org/gtr/api/organisations/950B...,1,"{""links"": {""link"": [{""href"": ""http://gtr.ukri....",GTR,BRONZE,2026-07-30 16:18:30,2026-07-30 16:18:30,None,None
1,2,095A3C0E-E4E6-4D44-945E-0036610164BA,http://gtr.ukri.org/gtr/api/projects/095A3C0E-...,Novel Time-Resolved Thermal Imaging: AlGaN/GaN...,Closed,None,Research Grant,The increasing complexity of tasks required by...,None,None,...,None,http://gtr.ukri.org/gtr/api/organisations/2533...,1,"{""links"": {""link"": [{""href"": ""http://gtr.ukri....",GTR,BRONZE,2026-07-30 16:18:30,2026-07-30 16:18:30,None,None
2,3,09E9D9A8-C863-48AA-B42B-004D4B4A472D,http://gtr.ukri.org/gtr/api/projects/09E9D9A8-...,Open Access Block Award 2023 - Armagh Observat...,Closed,None,Research Grant,Abstracts are not currently available in GtR f...,None,None,...,None,http://gtr.ukri.org/gtr/api/organisations/9ADA...,1,"{""links"": {""link"": [{""href"": ""http://gtr.ukri....",GTR,BRONZE,2026-07-30 16:18:30,2026-07-30 16:18:30,None,None



Gateway to Research Silver preview
Rows: 14 | Columns: 25


,record_id,project_id,project_url,title,status,grant_reference,grant_category,abstract_text,start_date,end_date,...,API_Page,raw_project_json,source_system,medallion_layer,inserted_date,modified_date,matching_method,matching_accuracy_percent,lead_organisation,lead_organisation_clean
8,9,05EDCC8E-392F-4032-B566-0148F8E61927,http://gtr.ukri.org/gtr/api/projects/05EDCC8E-...,Assessing hurricane damage and species vulnera...,Active,None,Research Grant,"On 28 November 2025, Hurricane Melissa, a Cate...",NaT,NaT,...,1,"{""links"": {""link"": [{""href"": ""http://gtr.ukri....",GTR,SILVER,2026-07-30 16:18:46,2026-07-30 16:18:46,None,None,NORTHUMBRIA UNIVERSITY,NORTHUMBRIA UNIVERSITY
17,18,0CABBA18-5FC8-4B1F-BE98-00848D0342B4,http://gtr.ukri.org/gtr/api/projects/0CABBA18-...,ERDERA - EUROPEAN RARE DISEASES RESEARCH ALLIANCE,Active,None,EU-Funded,Abstracts are not currently available in GtR f...,NaT,NaT,...,1,"{""links"": {""link"": [{""href"": ""http://gtr.ukri....",GTR,SILVER,2026-07-30 16:18:46,2026-07-30 16:18:46,None,None,NEWCASTLE UNIVERSITY,NEWCASTLE UNIVERSITY
19,20,0CF15F24-E99E-4496-83E0-0191A65E11DB,http://gtr.ukri.org/gtr/api/projects/0CF15F24-...,Predicting Biological Carbon in the Ocean Glob...,Active,None,Fellowship,Emissions of carbon dioxide (CO2) from our soc...,NaT,NaT,...,1,"{""links"": {""link"": [{""href"": ""http://gtr.ukri....",GTR,SILVER,2026-07-30 16:18:46,2026-07-30 16:18:46,None,None,UNIVERSITY OF LIVERPOOL,UNIVERSITY OF LIVERPOOL



Gateway to Research automated validation evidence
Total comparison rows: 11

Automated validation status counts:
auto_validation_status
ACCEPTED                        9
EXCLUDED_SINGLE_METHOD          1
EXCLUDED_METHOD_DISAGREEMENT    1

Dashboard eligible rows: 9
Excluded / not eligible rows: 2
Fuzzy and TF-IDF agreement rate: 81.82 %

Pipeline summary completed.


In [13]:
# SECTION 9 - SQL SERVER UPLOAD

# SAFE SQL NOTE:
# fast_executemany is disabled because pyodbc can create string-buffer truncation
# when long SponsorRoute values are inserted into SQL Server.


# Automated validation evidence is stored in dbo.MatchComparison and dbo.vw_AutomatedValidationEvidence.
# DataLoadAudit stores CF and GTR file timestamps; Home Office and runtime-log CSV timestamps remain visible in Google Drive.

if RUN_SQL_UPLOAD:
    sql_total_start = time.perf_counter()
    from sqlalchemy import create_engine, text
    from urllib.parse import quote_plus

    connection_string = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DATABASE};"
        f"UID={SQL_USERNAME};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
    )

    engine = create_engine(
        "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string),
        fast_executemany=False,
        isolation_level="AUTOCOMMIT"
    )

    REQUIRED_TABLES = [
        "DataLoadAudit",
        "DimSponsorOrganisation",
        "Gold_ContractsFinder",
        "Gold_GtRResearch",
        "MatchComparison"
    ]

    def sql_table_exists(conn, table_name):
        return conn.execute(
            text("SELECT CASE WHEN OBJECT_ID(:table_name, 'U') IS NULL THEN 0 ELSE 1 END"),
            {"table_name": f"dbo.{table_name}"}
        ).scalar() == 1

    def clean_target_value(value):
        if pd.isna(value):
            return None
        return value

    def find_column(df, candidates):
        if df is None or df.empty:
            return None

        def norm(x):
            return re.sub(r"[^a-z0-9]", "", str(x).lower())

        lookup = {norm(c): c for c in df.columns}
        for candidate in candidates:
            direct = candidate
            if direct in df.columns:
                return direct
            key = norm(candidate)
            if key in lookup:
                return lookup[key]
        return None

    def series_or_none(df, candidates):
        col = find_column(df, candidates)
        if col is None:
            return pd.Series([None] * len(df), index=df.index)
        return df[col]

    def method_to_sql(value):
        value = str(value).strip().upper()
        if "EXACT" in value:
            return "Exact"
        if "ML" in value or "TFIDF" in value or "COSINE" in value:
            return "ML"
        return "Fuzzy"

    def prepare_gold_for_target(df_gold, source_system, sponsor_key_map):
        if df_gold is None or df_gold.empty:
            if source_system == "CF":
                return pd.DataFrame(columns=[
                    "SponsorKey", "Contract_Title", "Contract_Value", "Supplier_Name",
                    "Award_Date", "matched_sponsor", "Town_City", "Route",
                    "MatchMethod", "MatchingAccuracyPercent", "InsertedDate", "ModifiedDate"
                ])
            return pd.DataFrame(columns=[
                "SponsorKey", "project_id", "title", "status", "start_date",
                "lead_organisation", "matched_sponsor", "MatchMethod",
                "MatchingAccuracyPercent", "InsertedDate", "ModifiedDate"
            ])

        df = df_gold.copy()
        official_col = find_column(df, [
            "sponsor_Organisation Name", "sponsor_Organisation_Name",
            "sponsor_organisation_name", "matched_sponsor",
            "Organisation Name", "Organisation_Name"
        ])
        town_col = find_column(df, ["sponsor_Town/City", "sponsor_Town_City", "Town_City", "TownCity", "Town/City"])
        route_col = find_column(df, ["sponsor_Route", "Route", "SponsorRoute"])
        match_method_col = find_column(df, ["matching_method", "MatchMethod", "method"])
        score_col = find_column(df, [
            "matching_accuracy_percent", "MatchingAccuracyPercent",
            "matching_confidence_percent", "match_score", "score_percent",
            "fuzzy_score_percent", "ml_score_percent", "FuzzyScorePercent", "MLScorePercent"
        ])

        if official_col is not None:
            matched_sponsor = df[official_col].fillna(df.get("matched_sponsor_clean", ""))
        else:
            matched_sponsor = df.get("matched_sponsor_clean", pd.Series([None] * len(df), index=df.index))

        common = pd.DataFrame(index=df.index)
        common["matched_sponsor"] = matched_sponsor.astype(str).replace({"nan": None, "": None})
        common["SponsorKey"] = common["matched_sponsor"].map(sponsor_key_map)
        common["MatchMethod"] = series_or_none(df, [match_method_col if match_method_col else "matching_method"]).apply(method_to_sql)

        score_values = pd.to_numeric(
            series_or_none(df, [score_col if score_col else "matching_accuracy_percent"]),
            errors="coerce"
        )

        source_clean = series_or_none(df, ["source_organisation_clean"]).astype(str)
        matched_clean = series_or_none(df, ["matched_sponsor_clean"]).astype(str)
        exact_like = (source_clean == matched_clean) | (common["MatchMethod"] == "Exact")
        score_values = score_values.mask(score_values.isna() & exact_like, 100.0)

        common["MatchingAccuracyPercent"] = score_values
        common["InsertedDate"] = current_timestamp()
        common["ModifiedDate"] = current_timestamp()

        if source_system == "CF":
            out = pd.DataFrame(index=df.index)
            out["SponsorKey"] = common["SponsorKey"]
            out["Contract_Title"] = series_or_none(df, ["Contract_Title"])
            out["Contract_Value"] = pd.to_numeric(series_or_none(df, ["Contract_Value"]), errors="coerce")
            out["Supplier_Name"] = series_or_none(df, ["Supplier_Name"])
            out["Award_Date"] = series_or_none(df, ["Award_Date"])
            out["matched_sponsor"] = common["matched_sponsor"]
            out["Town_City"] = series_or_none(df, [town_col if town_col else "Town_City"])
            out["Route"] = series_or_none(df, [route_col if route_col else "Route"])
            out["MatchMethod"] = common["MatchMethod"]
            out["MatchingAccuracyPercent"] = common["MatchingAccuracyPercent"]
            out["InsertedDate"] = common["InsertedDate"]
            out["ModifiedDate"] = common["ModifiedDate"]
            return out

        out = pd.DataFrame(index=df.index)
        out["SponsorKey"] = common["SponsorKey"]
        out["project_id"] = series_or_none(df, ["project_id"])
        out["title"] = series_or_none(df, ["title"])
        out["status"] = series_or_none(df, ["status"])
        out["start_date"] = series_or_none(df, ["start_date"])
        out["lead_organisation"] = series_or_none(df, ["lead_organisation"])
        out["matched_sponsor"] = common["matched_sponsor"]
        out["MatchMethod"] = common["MatchMethod"]
        out["MatchingAccuracyPercent"] = common["MatchingAccuracyPercent"]
        out["InsertedDate"] = common["InsertedDate"]
        out["ModifiedDate"] = common["ModifiedDate"]
        return out

    def build_sponsor_dimension(gold_frames):
        rows = []
        for df in gold_frames:
            if df is None or df.empty:
                continue

            official_col = find_column(df, [
                "sponsor_Organisation Name", "sponsor_Organisation_Name",
                "sponsor_organisation_name", "matched_sponsor"
            ])
            town_col = find_column(df, ["sponsor_Town/City", "sponsor_Town_City", "Town_City", "TownCity"])
            route_col = find_column(df, ["sponsor_Route", "Route", "SponsorRoute"])

            if official_col is None:
                official = df.get("matched_sponsor_clean", pd.Series([None] * len(df), index=df.index))
            else:
                official = df[official_col].fillna(df.get("matched_sponsor_clean", ""))

            temp = pd.DataFrame({
                "MatchedSponsor": official,
                "TownCity": series_or_none(df, [town_col if town_col else "TownCity"]),
                "SponsorRoute": series_or_none(df, [route_col if route_col else "SponsorRoute"])
            })
            rows.append(temp)

        if not rows:
            return pd.DataFrame(columns=["MatchedSponsor", "TownCity", "SponsorRoute", "InsertedDate", "ModifiedDate"])

        dim = pd.concat(rows, ignore_index=True)
        dim["MatchedSponsor"] = dim["MatchedSponsor"].astype(str).replace({"nan": None, "": None})
        dim = dim.dropna(subset=["MatchedSponsor"]).drop_duplicates(subset=["MatchedSponsor"])
        dim["InsertedDate"] = current_timestamp()
        dim["ModifiedDate"] = current_timestamp()
        return dim[["MatchedSponsor", "TownCity", "SponsorRoute", "InsertedDate", "ModifiedDate"]]

    def prepare_comparison_for_target(df_comparison, source_system):
        if df_comparison is None or df_comparison.empty:
            return pd.DataFrame(columns=[
                "SourceSystem", "SourceRecordKey", "SourceOrganisationName",
                "FuzzyMatchedSponsor", "FuzzyScorePercent", "MLMatchedSponsor",
                "MLScorePercent", "MethodsAgree", "RecommendedMethod",
                "RecommendedMatchedSponsor", "RecommendedScorePercent",
                "AutoValidationStatus", "AutoValidationReason", "IsDashboardEligible",
                "FuzzyCorrect", "MLCorrect", "ValidationNotes",
                "InsertedDate", "ModifiedDate"
            ])

        df = df_comparison.copy()
        source_name = "ContractsFinder" if source_system == "CF" else "GtRResearch"

        source_org = series_or_none(df, ["source_organisation_clean", "SourceOrganisationName", "SourceRecordKey"])
        fuzzy_match = series_or_none(df, ["fuzzy_match", "FuzzyMatchedSponsor", "fuzzy_matched_sponsor"])
        ml_match = series_or_none(df, ["ml_match", "MLMatchedSponsor", "ml_matched_sponsor"])

        fuzzy_score = pd.to_numeric(
            series_or_none(df, ["fuzzy_score_percent", "FuzzyScorePercent", "fuzzy_score", "fuzzy_match_score"]),
            errors="coerce"
        )
        ml_score = pd.to_numeric(
            series_or_none(df, ["ml_score_percent", "MLScorePercent", "ml_score", "ml_match_score"]),
            errors="coerce"
        )

        source_as_text = source_org.astype(str)
        fuzzy_score = fuzzy_score.mask(fuzzy_score.isna() & fuzzy_match.notna() & (source_as_text == fuzzy_match.astype(str)), 100.0)
        ml_score = ml_score.mask(ml_score.isna() & ml_match.notna() & (source_as_text == ml_match.astype(str)), 100.0)

        methods_agree = (
            fuzzy_match.fillna("").astype(str) == ml_match.fillna("").astype(str)
        ) & fuzzy_match.notna() & ml_match.notna()

        recommended_method = series_or_none(df, ["recommended_method", "RecommendedMethod"])

        # SQL-safe method labels.
        # The database CHECK constraint for MatchComparison.RecommendedMethod only accepts the
        # core method names used for comparison evidence. Exact/agreement cases are still recorded
        # through MethodsAgree, RecommendedScorePercent, AutoValidationStatus and AutoValidationReason.
        def sql_safe_recommended_method(value):
            if pd.isna(value):
                return None

            v = str(value).strip()
            lower_v = v.lower()

            if v in ["Fuzzy", "ML"]:
                return v

            if lower_v in ["exact", "fuzzy+ml", "fuzzy + ml", "both", "agreement", "methodsagree"]:
                return "Fuzzy"

            if "ml" in lower_v and "fuzzy" not in lower_v:
                return "ML"

            if "fuzzy" in lower_v or "exact" in lower_v:
                return "Fuzzy"

            return None

        recommended_method = recommended_method.apply(sql_safe_recommended_method)

        recommended_match = series_or_none(df, ["recommended_match", "RecommendedMatchedSponsor"])
        recommended_score = pd.to_numeric(
            series_or_none(df, ["recommended_score_percent", "RecommendedScorePercent"]),
            errors="coerce"
        )

        # SQL-safe recommended score.
        # The database CHECK constraint requires RecommendedScorePercent to be within 0-100.
        # Some excluded rows have no accepted recommendation score, so they are set to 0.
        # Accepted rows use the available fuzzy/ML score when the recommended score is missing.
        recommended_score = recommended_score.fillna(
            pd.concat([fuzzy_score, ml_score], axis=1).max(axis=1)
        )
        recommended_score = recommended_score.fillna(0).clip(lower=0, upper=100).round(2)

        auto_status = series_or_none(df, ["auto_validation_status", "AutoValidationStatus"])
        auto_reason = series_or_none(df, ["auto_validation_reason", "AutoValidationReason"])
        dashboard_eligible = pd.to_numeric(
            series_or_none(df, ["dashboard_eligible", "IsDashboardEligible"]),
            errors="coerce"
        ).fillna(0).astype(int)

        out = pd.DataFrame({
            "SourceSystem": source_name,
            "SourceRecordKey": source_org,
            "SourceOrganisationName": source_org,
            "FuzzyMatchedSponsor": fuzzy_match,
            "FuzzyScorePercent": fuzzy_score,
            "MLMatchedSponsor": ml_match,
            "MLScorePercent": ml_score,
            "MethodsAgree": methods_agree.astype(int),
            "RecommendedMethod": recommended_method,
            "RecommendedMatchedSponsor": recommended_match,
            "RecommendedScorePercent": recommended_score,
            "AutoValidationStatus": auto_status,
            "AutoValidationReason": auto_reason,
            "IsDashboardEligible": dashboard_eligible,
            "FuzzyCorrect": None,
            "MLCorrect": None,
            "ValidationNotes": auto_reason,
            "InsertedDate": current_timestamp(),
            "ModifiedDate": current_timestamp()
        })
        return out


    def filter_gold_by_dashboard_eligibility(df_gold, comparison_df):
        if df_gold is None or df_gold.empty:
            return df_gold
        if comparison_df is None or comparison_df.empty:
            return df_gold.head(0).copy()
        if "dashboard_eligible" not in comparison_df.columns:
            return df_gold

        eligible_orgs = set(
            comparison_df.loc[
                comparison_df["dashboard_eligible"].astype(str).isin(["1", "True", "true"]),
                "source_organisation_clean"
            ].dropna().astype(str)
        )

        if "source_organisation_clean" not in df_gold.columns:
            return df_gold

        return df_gold[df_gold["source_organisation_clean"].astype(str).isin(eligible_orgs)].copy()


    def build_recommended_gold(fuzzy_df, ml_df, comparison_df):
        """Build one dashboard-ready Gold dataset using accepted automated validation results only."""
        if comparison_df is None or comparison_df.empty:
            return pd.DataFrame()
        if "dashboard_eligible" not in comparison_df.columns:
            return pd.DataFrame()

        accepted = comparison_df[comparison_df["dashboard_eligible"].astype(str).isin(["1", "True", "true"])].copy()
        if accepted.empty:
            return pd.DataFrame()

        # Use fuzzy rows as the main record source because accepted records require method agreement or exact match.
        # Fall back to ML rows if fuzzy rows are not available.
        base = fuzzy_df.copy() if fuzzy_df is not None and not fuzzy_df.empty else pd.DataFrame()
        if base.empty and ml_df is not None and not ml_df.empty:
            base = ml_df.copy()
        if base.empty or "source_organisation_clean" not in base.columns:
            return pd.DataFrame()

        accepted_cols = accepted[[
            "source_organisation_clean",
            "recommended_method",
            "recommended_match",
            "recommended_score_percent",
            "auto_validation_status",
            "auto_validation_reason",
            "dashboard_eligible"
        ]].copy()

        out = base.merge(accepted_cols, on="source_organisation_clean", how="inner")
        if out.empty:
            return out

        out["matched_sponsor_clean"] = out["recommended_match"]
        out["matching_method"] = out["recommended_method"]
        out["matching_accuracy_percent"] = out["recommended_score_percent"]
        return out


    def build_audit_dataframe(file_paths):
        rows = []
        # Only these two SourceSystem values are uploaded to DataLoadAudit.
        # Some database versions have a CHECK constraint that rejects other values.
        source_lookup = {
            "CF": "ContractsFinder",
            "GTR": "GtRResearch"
        }
        layer_lookup = {
            "BRONZE": "Bronze",
            "SILVER": "Silver",
            "GOLD": "Gold",
            "COMPARISON": "Comparison"
        }
        method_lookup = {"FUZ": "Fuzzy", "ML": "ML"}

        for file_path in file_paths:
            file_name = os.path.basename(file_path)
            name_without_ext = os.path.splitext(file_name)[0]
            parts = name_without_ext.split("_")

            source_code = parts[0] if len(parts) > 0 else ""
            layer_code = parts[1] if len(parts) > 1 else ""
            method_code = parts[2] if len(parts) > 2 else None

            # HOME_SPONSOR and LD6053_RUNTIME_LOG are kept in Google Drive as evidence,
            # but they are not inserted into DataLoadAudit to avoid SQL CHECK constraint errors.
            if source_code not in source_lookup:
                continue

            try:
                row_count = len(pd.read_csv(file_path, low_memory=False))
            except Exception:
                row_count = 0

            rows.append({
                "RunDate": RUN_DATE,
                "FileName": file_name,
                "FilePath": file_path,
                "SourceSystem": source_lookup[source_code],
                "LayerName": layer_lookup.get(layer_code, layer_code.title()),
                "MatchMethod": method_lookup.get(method_code, None),
                "RowsLoaded": int(row_count),
                "InsertedDate": current_timestamp(),
                "ModifiedDate": current_timestamp()
            })

        return pd.DataFrame(rows)

    def sql_table_metadata(table_name):
        with engine.connect() as conn:
            rows = conn.execute(
                text("""
                    SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
                    FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = 'dbo' AND TABLE_NAME = :table_name
                    ORDER BY ORDINAL_POSITION
                """),
                {"table_name": table_name}
            ).fetchall()
        return pd.DataFrame(rows, columns=["COLUMN_NAME", "DATA_TYPE", "CHARACTER_MAXIMUM_LENGTH"])


    def fit_dataframe_to_sql_table(table_name, df):
        """Keep only target columns and trim long text to the SQL column size."""
        df = prepare_for_sql(df)
        meta = sql_table_metadata(table_name)

        if meta.empty:
            return df

        target_columns = meta["COLUMN_NAME"].tolist()
        dropped_columns = [c for c in df.columns if c not in target_columns]
        keep_columns = [c for c in target_columns if c in df.columns]

        if dropped_columns:
            print(f"Columns not found in dbo.{table_name} and not uploaded:", dropped_columns)

        df = df[keep_columns].copy()

        for _, row in meta.iterrows():
            col = row["COLUMN_NAME"]
            data_type = str(row["DATA_TYPE"]).lower()
            max_len = row["CHARACTER_MAXIMUM_LENGTH"]

            if col not in df.columns:
                continue

            if data_type in ["varchar", "nvarchar", "char", "nchar"] and pd.notna(max_len):
                max_len = int(max_len)
                if max_len > 0:
                    df[col] = df[col].apply(
                        lambda x: x[:max_len] if isinstance(x, str) and len(x) > max_len else x
                    )

        return df


    def insert_dataframe(table_name, df):
        step_start = time.perf_counter()

        try:
            if df is None or df.empty:
                print(f"No rows to insert into dbo.{table_name}")
                log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "SUCCESS", 0, "No rows to insert")
                return

            df = fit_dataframe_to_sql_table(table_name, df)

            if df.empty:
                print(f"No matching columns/rows to insert into dbo.{table_name}")
                log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "SUCCESS", 0, "No matching rows/columns")
                return

            for start in range(0, len(df), SQL_INSERT_CHUNK_SIZE):
                end = min(start + SQL_INSERT_CHUNK_SIZE, len(df))
                chunk = df.iloc[start:end]
                chunk.to_sql(
                    table_name,
                    con=engine,
                    schema="dbo",
                    if_exists="append",
                    index=False,
                    chunksize=len(chunk),
                    method=None
                )

            print(f"Inserted {len(df):,} rows into dbo.{table_name}")
            log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "SUCCESS", len(df), "")

        except Exception as e:
            log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "FAILED", None, str(e))
            save_runtime_log()
            raise


    clear_start = time.perf_counter()
    try:
        with engine.begin() as conn:
            missing = [t for t in REQUIRED_TABLES if not sql_table_exists(conn, t)]
            if missing:
                raise RuntimeError(
                    "These SQL tables are missing: " + ", ".join(missing) +
                    ". Run the database creation script first, then run this notebook again."
                )

            # Delete child/detail tables before dimension/audit tables.
            conn.execute(text("DELETE FROM dbo.MatchComparison;"))
            conn.execute(text("DELETE FROM dbo.Gold_ContractsFinder;"))
            conn.execute(text("DELETE FROM dbo.Gold_GtRResearch;"))
            conn.execute(text("DELETE FROM dbo.DimSponsorOrganisation;"))
            conn.execute(text("DELETE FROM dbo.DataLoadAudit;"))
            print("Cleared target SQL tables.")
        log_runtime_step("SQL clear target tables", clear_start, "SUCCESS", None, "")
    except Exception as e:
        log_runtime_step("SQL clear target tables", clear_start, "FAILED", None, str(e))
        save_runtime_log()
        raise

    cf_comparison_raw = globals().get("cf_comparison", pd.DataFrame())
    gtr_comparison_raw = globals().get("gtr_comparison", pd.DataFrame())

    cf_gold_sql = build_recommended_gold(
        globals().get("cf_gold_fuzzy", pd.DataFrame()),
        globals().get("cf_gold_ml", pd.DataFrame()),
        cf_comparison_raw
    )
    gtr_gold_sql = build_recommended_gold(
        globals().get("gtr_gold_fuzzy", pd.DataFrame()),
        globals().get("gtr_gold_ml", pd.DataFrame()),
        gtr_comparison_raw
    )

    sponsor_dim = build_sponsor_dimension([cf_gold_sql, gtr_gold_sql])

    insert_dataframe("DimSponsorOrganisation", sponsor_dim)

    with engine.connect() as conn:
        sponsor_key_df = pd.read_sql(
            "SELECT SponsorKey, MatchedSponsor FROM dbo.DimSponsorOrganisation;",
            conn
        )
    sponsor_key_map = dict(zip(sponsor_key_df["MatchedSponsor"], sponsor_key_df["SponsorKey"]))

    cf_gold_combined = prepare_gold_for_target(cf_gold_sql, "CF", sponsor_key_map)
    gtr_gold_combined = prepare_gold_for_target(gtr_gold_sql, "GTR", sponsor_key_map)

    cf_comparison_sql = prepare_comparison_for_target(cf_comparison_raw, "CF")
    gtr_comparison_sql = prepare_comparison_for_target(gtr_comparison_raw, "GTR")
    comparison_combined = pd.concat([cf_comparison_sql, gtr_comparison_sql], ignore_index=True)

    # Final safety check before SQL upload.
    # This prevents SQL CHECK constraint failures if a future code change creates labels such as
    # Exact, Both or Fuzzy+ML in RecommendedMethod.
    if not comparison_combined.empty and "RecommendedMethod" in comparison_combined.columns:
        comparison_combined["RecommendedMethod"] = comparison_combined["RecommendedMethod"].apply(
            lambda x: (
                None if pd.isna(x)
                else "ML" if str(x).strip().lower() == "ml"
                else "Fuzzy" if str(x).strip().lower() in ["fuzzy", "exact", "fuzzy+ml", "fuzzy + ml", "both", "agreement", "methodsagree"]
                else "ML" if ("ml" in str(x).strip().lower() and "fuzzy" not in str(x).strip().lower())
                else "Fuzzy" if ("fuzzy" in str(x).strip().lower() or "exact" in str(x).strip().lower())
                else None
            )
        )

    # Final score safety check before SQL upload.
    if not comparison_combined.empty and "RecommendedScorePercent" in comparison_combined.columns:
        comparison_combined["RecommendedScorePercent"] = pd.to_numeric(
            comparison_combined["RecommendedScorePercent"],
            errors="coerce"
        ).fillna(0).clip(lower=0, upper=100).round(2)

    audit_df = build_audit_dataframe(created_files)

    insert_dataframe("DataLoadAudit", audit_df)
    insert_dataframe("Gold_ContractsFinder", cf_gold_combined)
    insert_dataframe("Gold_GtRResearch", gtr_gold_combined)
    insert_dataframe("MatchComparison", comparison_combined)

    print("Automated validation evidence is stored in MatchComparison.")

    print("\nSQL row-count check")
    with engine.connect() as conn:
        for table in ["DataLoadAudit", "DimSponsorOrganisation", "Gold_ContractsFinder", "Gold_GtRResearch", "MatchComparison"]:
            count = conn.execute(text(f"SELECT COUNT(*) FROM dbo.{table};")).scalar()
            print(f"{table}: {count:,}")

        def safe_view_count(view_name):
            try:
                count = conn.execute(text(f"SELECT COUNT(*) FROM dbo.{view_name};")).scalar()
                print(f"{view_name}: {count:,}")
            except Exception as e:
                print(f"{view_name}: not checked because the view is missing or needs updating.")
                print("Reason:", str(e).split("\n")[0])

        safe_view_count("vw_Dashboard_Overview")
        safe_view_count("vw_Dashboard_ContractsFinder")
        safe_view_count("vw_Dashboard_GtRResearch")
        safe_view_count("vw_MatchingComparisonSummary")
        safe_view_count("vw_AutomatedValidationEvidence")

    log_runtime_step("SQL upload total", sql_total_start, "SUCCESS", None, "")
    save_runtime_log()
    print("\nSQL upload completed.")
else:
    print("SQL upload skipped because RUN_SQL_UPLOAD is False.")
    save_runtime_log()


Cleared target SQL tables.
[TIME] SQL clear target tables: 1.17 seconds | SUCCESS
Inserted 296 rows into dbo.DimSponsorOrganisation
[TIME] SQL insert dbo.DimSponsorOrganisation: 0.63 seconds | SUCCESS
Inserted 10 rows into dbo.DataLoadAudit
[TIME] SQL insert dbo.DataLoadAudit: 0.4 seconds | SUCCESS
Inserted 404 rows into dbo.Gold_ContractsFinder
[TIME] SQL insert dbo.Gold_ContractsFinder: 0.54 seconds | SUCCESS
Inserted 11 rows into dbo.Gold_GtRResearch
[TIME] SQL insert dbo.Gold_GtRResearch: 0.31 seconds | SUCCESS
Inserted 623 rows into dbo.MatchComparison
[TIME] SQL insert dbo.MatchComparison: 2.47 seconds | SUCCESS
Automated validation evidence is stored in MatchComparison.

SQL row-count check
DataLoadAudit: 10
DimSponsorOrganisation: 296
Gold_ContractsFinder: 404
Gold_GtRResearch: 11
MatchComparison: 623
vw_Dashboard_Overview: 415
vw_Dashboard_ContractsFinder: 404
vw_Dashboard_GtRResearch: 11
vw_MatchingComparisonSummary: 2
vw_AutomatedValidationEvidence: 7
[TIME] SQL upload total